# **Modelo LightGCN sin multimodalidad**
### Proyecto Hito 3
### Sistemas Recomendadores IIC3633-1 2025-2
### **Grupo 3:** 

- Nicolás Antonio Bueno Abett de la Torre 

- Felipe Andrés Fuentes González

- Jorge Andrés Jacque Palma

- Francisco Nicolás Solís Gormaz

## Índice

>[0- Instalación de librerías](#0--instalación-de-librerías)

>[1- Carga de datos](#1--carga-de-datos)

>[2- Definición del modelo, formateo de datos, y entrenamiento](#2--definición-del-modelo-formateo-de-datos-y-entrenamiento)

>[3- Generación de recomendaciones](#3--generación-de-recomendaciones)

>[4- Métricas](#4--métricas)

>[5- Referencias](#5--referencias)

## 0- Instalación de librerías

En caso de usar Colab, correr estas celdas. Si se corre en local, se deben tener exactamente las mismas versiones de las librerías indicadas en estas celdas. Las versiones de las librerías son:

- numpy: 1.25.0
- pandas: 2.2.2
- scipy: 1.10.1
- tqdm: 4.66.6
- torch: 2.5.1+cpu
- tensorboard: 2.12.3
- recbole: 1.2.1

In [ ]:
# !pip uninstall -y numpy
# !pip install numpy==1.25

In [ ]:
# !pip uninstall -y pandas
# !pip install pandas==2.2.2

In [ ]:
# !pip uninstall -y scipy
# !pip install scipy==1.10.1

In [ ]:
# !pip uninstall -y tqdm
# !pip install tqdm==4.66.6

In [ ]:
# !pip uninstall -y torch
# !pip uninstall -y torchvision
# !pip uninstall -y torchaudio
# !pip install torch==2.5.1+cpu --index-url https://download.pytorch.org/whl/cpu

In [ ]:
# #Necesario para RecBole, también instala tensorboard-data-server 0.7.2
# !pip uninstall -y tensorboard
# !pip install tensorboard==2.12.3

In [ ]:
# !pip uninstall -y recbole
# !pip install recbole==1.2.1

## 1- Carga de datos

Se leen los archivos de datos (entrenamiento, testeo, y validación) correspondientes al muestreo del dataset principal del proyecto ("Game Recommendations on Steam: A dataset of games, users and reviews for building recommendation systems". Anton Kozyriev, 2024. Kaggle.) [7] y se almacenan en un dataframe:

In [1]:
import pandas as pd

df_train = pd.read_csv('train_split.csv')
df_test = pd.read_csv('test_split.csv')
df_val = pd.read_csv('val_split.csv')

La estructura de estos archivos es:

In [2]:
df_train.head(5)

,app_id,helpful,funny,date,is_recommended,hours,user_id,review_id
0,322330,0,0,2019-07-02,True,67.5,731,33484606
1,433340,0,0,2020-01-24,True,32.3,731,26236770
2,394360,2,0,2020-04-20,True,403.7,731,25992499
3,4700,0,0,2020-04-21,True,683.5,731,9845461
4,246090,3,0,2015-02-25,False,7.9,3128,28148695


Dataframe de games.csv para mostrar nombres de juegos recomendados

In [ ]:
df_games = pd.read_csv('games.csv')

En este dataframe se guarda la información del archivo json de metadata de videojuegos:

In [2]:
games_metadata = pd.read_json('games_metadata.json', lines=True)
games_metadata.head(5)

,app_id,description,tags
0,13500,Enter the dark underworld of Prince of Persia ...,"[Action, Adventure, Parkour, Third Person, Gre..."
1,22364,,[Action]
2,113020,Monaco: What's Yours Is Mine is a single playe...,"[Co-op, Stealth, Indie, Heist, Local Co-Op, St..."
3,226560,Escape Dead Island is a Survival-Mystery adven...,"[Zombies, Adventure, Survival, Action, Third P..."
4,249050,Dungeon of the Endless is a Rogue-Like Dungeon...,"[Roguelike, Strategy, Tower Defense, Pixel Gra..."


Dado que el único feedback explícito que se posee en los datasets de interacciones es la columna "is_recommended", se añade una columna "rating" cuyo valor es 1 si "is_recommended" es "True", y su valor es 0 si "is_recommended" es "False". Esto se hace para cada uno de los df:

In [3]:
regla_rating = {True: 1, False: 0}

df_train['rating'] = df_train['is_recommended'].map(regla_rating)
df_test['rating'] = df_test['is_recommended'].map(regla_rating)
df_val['rating'] = df_val['is_recommended'].map(regla_rating)

El resultado es:

In [5]:
df_train.head(5)

,app_id,helpful,funny,date,is_recommended,hours,user_id,review_id,rating
0,322330,0,0,2019-07-02,True,67.5,731,33484606,1
1,433340,0,0,2020-01-24,True,32.3,731,26236770,1
2,394360,2,0,2020-04-20,True,403.7,731,25992499,1
3,4700,0,0,2020-04-21,True,683.5,731,9845461,1
4,246090,3,0,2015-02-25,False,7.9,3128,28148695,0


## 2- Definición del modelo LightGCN, formateo de datos, y entrenamiento

Se formatean los datos para la librería RecBole:

In [4]:
import os

#carpeta del dataset
dataset_name = "dataset_recbole"
dataset_dir = f"./dataset_recbole"
os.makedirs(dataset_dir, exist_ok=True)

#se formatean los dataframes para que tengan las columnas requeridas por RecBole
df_train_lightgcn = df_train[["user_id", "app_id", "rating"]].copy()
df_test_lightgcn = df_test[["user_id", "app_id", "rating"]].copy()
df_val_lightgcn = df_val[["user_id", "app_id", "rating"]].copy()
df_train_lightgcn.rename(columns={
    "user_id": "user_id:token",
    "app_id": "item_id:token",
    "rating": "rating:float"}, inplace=True)
df_test_lightgcn.rename(columns={
    "user_id": "user_id:token",
    "app_id": "item_id:token",
    "rating": "rating:float"}, inplace=True)
df_val_lightgcn.rename(columns={
    "user_id": "user_id:token",
    "app_id": "item_id:token",
    "rating": "rating:float"}, inplace=True)

#se guarda cada split como archivo .inter en la carpeta definida
df_train_lightgcn.to_csv(f"{dataset_dir}/{dataset_name}.train.inter", sep="\t", index=False)
df_test_lightgcn.to_csv(f"{dataset_dir}/{dataset_name}.test.inter", sep="\t", index=False)
df_val_lightgcn.to_csv(f"{dataset_dir}/{dataset_name}.valid.inter", sep="\t", index=False)

Se realizan iteraciones para encontrar la mejor combinación de valores de hiperparámetros del modelo. Se hace variar la cantidad de épocas (epochs), embedding size, número de capas (n_layers), y learning rate utilizando listas, eligiendo valores a partir de los recomendados por la librería RecBole y los estándar en investigación. Para esto se modifica el config_dict del modelo en donde se definen los valores de parámetros. Para cada iteración, se registran las métricas obtenidos por el modelo usando cada combinación de hiperparámetros, y finalmente se muestra la que obtuvo mejor valor. Se utilizará la combinación que alcance mejor valor en las métricas de NDCG@10, Precision@10, y Recall@10:

In [5]:
import os
import pandas as pd
import torch
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import LightGCN
from recbole.trainer import Trainer
from recbole.utils.case_study import full_sort_topk

#valores de parámetros a testear:
lista_epochs = [10, 20]
lista_embedding_size = [64, 128, 256]
lista_n_layers = [2, 3]
lista_learning_rate = [0.0005, 0.001, 0.002]

#registro de mejores valores de métricas y la combinación de hiperparámetros que las produjo
record_ndcg10 = 0.0
record_precision10 = 0.0
record_recall10 = 0.0

combinacion_record_ndcg10 = {}
combinacion_record_precision10 = {}
combinacion_record_recall10 = {}

#se ignora el warning "FutureWarning" para tener output más limpio:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

for epoch in lista_epochs:
    for emb_size in lista_embedding_size:
        for n_layer in lista_n_layers:
            for lr in lista_learning_rate:
                #diccionario de configuración requerido por RecBole para entrenar el modelo:
                config_dict_iteracion = {
                    "data_path": ".",
                    "model": "LightGCN",
                    "dataset": dataset_name,
                    "format": "custom",
                    "benchmark_filename" : ['train', 'valid', 'test'],
                    "dataset_file": {
                        "inter": {"train": f"{dataset_dir}/{dataset_name}.train.inter",
                            "valid": f"{dataset_dir}/{dataset_name}.valid.inter",
                            "test":  f"{dataset_dir}/{dataset_name}.test.inter"}
                    },
                    "field_separator": "\t",
                    "USER_ID_FIELD": "user_id",
                    "ITEM_ID_FIELD": "item_id",
                    "RATING_FIELD": "rating",
                    "load_col": {"inter": ["user_id", "item_id", "rating"]},
                    "field_type": {"user_id": "token","item_id": "token","rating": "float"},
                    "eval_args": {"mode": "full"},
                    "epochs": epoch,
                    "train_batch_size": 2048,
                    "eval_batch_size": 4096,
                    "embedding_size": emb_size,
                    "show_progress": True,
                    "learning_rate": lr,
                    "reg_weight": 1e-5,
                    "n_layers": n_layer,
                    "topk": [10],
                    "device": "cpu",
                    #umbral de rating para considerar un ítem como relevante.
                    #en este caso, todo ítem con rating >= 0.5 es considerado relevante.
                    "threshold": {"rating": 0.5} 
                }

                #se crea el objeto Config de RecBole que guarda toda la configuración
                # requerida por la librería, se carga el dataset, y se preparan los datos:
                config_iteracion = Config(config_dict=config_dict_iteracion)
                dataset = create_dataset(config_iteracion)
                train_data, valid_data, test_data = data_preparation(config_iteracion, dataset)

                #modelo LightGCN y el Trainer de RecBole:
                model_iteracion = LightGCN(config_iteracion, train_data.dataset).to(config_iteracion['device'])
                trainer_iteracion = Trainer(config_iteracion, model_iteracion)

                #entrenamiento
                trainer_iteracion.fit(train_data, valid_data)
                print()
                print("Entrenamiento finalizado iteración")

                test_result_iteracion = trainer_iteracion.evaluate(test_data)
                metricas_iteracion = dict(test_result_iteracion)

                if metricas_iteracion['ndcg@10'] > record_ndcg10:
                    record_ndcg10 = metricas_iteracion['ndcg@10']
                    combinacion_record_ndcg10 = {
                        "epochs": epoch,
                        "embedding_size": emb_size,
                        "n_layers": n_layer,
                        "learning_rate": lr
                    }

                if metricas_iteracion['precision@10'] > record_precision10:
                    record_precision10 = metricas_iteracion['precision@10']
                    combinacion_record_precision10 = {
                        "epochs": epoch,
                        "embedding_size": emb_size,
                        "n_layers": n_layer,
                        "learning_rate": lr
                    }
                
                if metricas_iteracion['recall@10'] > record_recall10:
                    record_recall10 = metricas_iteracion['recall@10']
                    combinacion_record_recall10 = {
                        "epochs": epoch,
                        "embedding_size": emb_size,
                        "n_layers": n_layer,
                        "learning_rate": lr
                    }

print("Mejores valores y combinaciones de hiperparámetros:")
print()
print("NDCG@10:", record_ndcg10)
print("Combinación de hiperparámetros:", combinacion_record_ndcg10)
print()
print("Precision@10:", record_precision10)
print("Combinación de hiperparámetros:", combinacion_record_precision10)
print()
print("Recall@10:", record_recall10)
print("Combinación de hiperparámetros:", combinacion_record_recall10)

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

c:\Users\felip\AppData\Local\Programs\Python\Python311\Lib\site-packages\recbole\model\general_recommender\lightgcn.py:125: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_new.cpp:653.)
  SparseL = torch.sparse.FloatTensor(i, data, torch.Size(L.shape))



Entrenamiento finalizado iteración


KeyboardInterrupt: 

Ahora, para evitar ejecutar el código de arriba cada vez (pues se demora apróx. 70 min) se registran aquí los valores de hiperparámetros que obtuvieron las mejores métricas, para usarlos en el  modelo final:

In [6]:
mejor_epoch = 20
mejor_embedding_size = 256
mejor_n_layers = 2
mejor_learning_rate = 0.002

Ahora, utilizando la mejor combinación de parámetros en base a los valores de métricas calculadas por la librería, se define el modelo final que se usará y se entrena:

In [7]:
import os
import pandas as pd
import torch
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import LightGCN
from recbole.trainer import Trainer
from recbole.utils.case_study import full_sort_topk

#diccionario de configuración requerido por RecBole para entrenar el modelo:
config_dict = {
    "data_path": ".",
    "model": "LightGCN",
    "dataset": dataset_name,
    "format": "custom",
    "benchmark_filename" : ['train', 'valid', 'test'],
    "dataset_file": {
        "inter": {
            "train": f"{dataset_dir}/{dataset_name}.train.inter",
            "valid": f"{dataset_dir}/{dataset_name}.valid.inter",
            "test":  f"{dataset_dir}/{dataset_name}.test.inter"
        }
    },
    "field_separator": "\t",
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "RATING_FIELD": "rating",
    "load_col": {"inter": ["user_id", "item_id", "rating"]},
    "field_type": {
        "user_id": "token",
        "item_id": "token",
        "rating": "float"
    },
    "eval_args": {"mode": "full"},
    "epochs": mejor_epoch,
    "train_batch_size": 2048,
    "eval_batch_size": 4096,
    "embedding_size": mejor_embedding_size,
    "show_progress": True,
    "learning_rate": mejor_learning_rate,
    "reg_weight": 1e-5,
    "n_layers": mejor_n_layers,
    "topk": [10],
    "device": "cpu",
    #umbral de rating para considerar un ítem como relevante.
    #en este caso, todo ítem con rating >= 0.5 es considerado relevante.
    "threshold": {"rating": 0.5} 
}

#se crea el objeto Config de RecBole que guarda toda la configuración
# requerida por la librería, se carga el dataset, y se preparan los datos:

config = Config(config_dict=config_dict)
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

#modelo LightGCN y el Trainer de RecBole:
model = LightGCN(config, train_data.dataset).to(config['device'])
trainer = Trainer(config, model)

#entrenamiento
trainer.fit(train_data, valid_data)
print()
print("Entrenamiento finalizado")


Entrenamiento finalizado


## 3- Generación de recomendaciones

Ahora, a partir del entrenamiento anterior se obtienen las listas de recomendación top 10 para cada usuario. Se comienza usando las ids internas de la librería, para luego convertirlas a las originales del dataset:

In [8]:
#se guardan las ids internas de los usuarios presentes en el dataset,
# evitando la id = 0 que se asocia al ['PAD'] que usa internamente RecBole
dataset_obj = test_data.dataset
uid_series = [uid for uid in range(1, dataset_obj.user_num) 
              if test_data.uid2history_item[uid] is not None]

#se obtienen las listas de recomendación con todos los ítems para cada usuario
# se usa topk = número total de ítems (incluyendo el 'PAD' con id = 0)
topk = 10
topk_result = full_sort_topk(uid_series, model, test_data, k=topk, device=config['device'])

Se guardan en un diccionario las recomendaciones con las ids de usuario e ítem originales, convirtiendo primero las ids internas de RecBole a las originales del dataset. El diccionario tiene como llaves la id del usuario y su valor es una lista con las 10 id de ítems recomendados:

In [9]:
#se convierten las ids de usuarios internas de RecBole a las originales del dataset
user_original_ids = dataset_obj.id2token('user_id', uid_series)

#se convierten las ids de ítems internas de RecBole a las originales del dataset
topk_items_original = [
    dataset_obj.id2token('item_id', topk_result.indices[i])
    for i in range(len(uid_series))
]

#diccionario final de recomendaciones:

#con ids como strings:
# recommendations_dict = {int(user_id): items for user_id, items in zip(user_original_ids, topk_items_original)}

#con ids como enteros:
top10_lightgcn = {int(user_id): [int(item) for item in items] for user_id, items in zip(user_original_ids, topk_items_original)}


print("Ejemplo de 5 usuarios y sus recomendaciones:")
for user_id, items in list(top10_lightgcn.items())[:5]:
    print(f"Usuario {user_id} → {items}")

Ejemplo de 5 usuarios y sus recomendaciones:
Usuario 731 → [440, 294100, 4000, 431960, 427520, 236390, 107410, 244850, 48700, 304930]
Usuario 3128 → [275850, 261550, 233860, 427520, 294100, 392160, 1124300, 1149460, 529180, 1086940]
Usuario 4232 → [220, 218620, 440, 550, 12210, 238320, 444090, 359550, 286690, 271590]
Usuario 8297 → [1091500, 431960, 945360, 1172620, 629730, 1085660, 275850, 242760, 1174180, 526870]
Usuario 9905 → [304930, 444090, 4000, 105600, 945360, 620, 550, 218620, 49520, 700330]


Usuarios con recomendaciones (en total el dataset tiene 3000):

In [ ]:
len(top10_lightgcn)

9906

Ejemplo recomendación usuario con ID = 731:

In [ ]:
print("Ejemplo de usuario ID = 781 y sus recomendaciones:")
print("IDs videojuegos recomendados:")
print(top10_lightgcn[731])
print("Títulos videojuegos recomendados:")
print([df_games[df_games["app_id"] == item]["title"].values[0] for item in top10_lightgcn[731]])

Ejemplo de usuario ID = 781 y sus recomendaciones:
IDs videojuegos recomendados:
[440, 294100, 4000, 431960, 427520, 236390, 107410, 244850, 48700, 304930]
Títulos videojuegos recomendados:
['Team Fortress 2', 'RimWorld', "Garry's Mod", 'Wallpaper Engine', 'Factorio', 'War Thunder', 'Arma 3', 'Space Engineers', 'Mount & Blade: Warband', 'Unturned']


## 4- Métricas

Para facilitar el cálculo, se crea el diccionario dict_items_relevantes que tiene como llave la id de usuario (para cada uno de los que tienen una recomendación top 10) y el valor es el set de ítems relevantes del dataset de test:

In [14]:
dic_items_relevantes = {}

for user_id in top10_lightgcn.keys():
    relevantes_usuario = set(df_test[(df_test["user_id"] == user_id) & (df_test["rating"] >= 0.5)]["app_id"])
    dic_items_relevantes[user_id] = relevantes_usuario

Cálculo NDCG@10:

In [15]:
import numpy as np

def ndcg_at_k(items_recomendados, items_relevantes, k=10):

    dcg = 0.0
    for i, item in enumerate(items_recomendados[:k]):
        if item in items_relevantes:
            dcg += 1 / np.log2(i + 2)

    ideal_hits = min(len(items_relevantes), k)
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_hits))

    if idcg == 0:
        return 0.0

    return dcg / idcg

ndcg_scores = {}
for user_id, items in top10_lightgcn.items():

    items_relevantes_usuario = dic_items_relevantes[user_id]

    ndcg_scores[user_id] = ndcg_at_k(items, items_relevantes_usuario, k=10)

ndcg_lightgcn = np.mean(list(ndcg_scores.values()))
print(f"NDCG@10: {ndcg_lightgcn:.4f}")

NDCG@10: 0.0279


Cálculo de Precision@10:

In [16]:
def precision_at_k(topk_dict, relevantes_dict, k=10):
    precisions = []

    for user_id, relevants in relevantes_dict.items():

        if user_id not in topk_dict:
            continue

        topk_items = topk_dict[user_id][:k]

        interseccion = len(set(topk_items) & relevants)

        precision_u = interseccion / k
        precisions.append(precision_u)

    return sum(precisions) / len(precisions) if precisions else 0.0

precision_lightgcn = precision_at_k(top10_lightgcn, dic_items_relevantes, k=10)
print(f"Precision@10: {precision_lightgcn:.4f}")


Precision@10: 0.0065


Cálculo de Recall@10:

In [17]:
def recall_at_k(topk_dict, relevantes_dict, k=10):
    recalls = []

    for user_id, relevants in relevantes_dict.items():

        if user_id not in topk_dict or len(relevants) == 0:
            continue 

        topk_items = topk_dict[user_id][:k]

        interseccion = len(set(topk_items) & relevants)

        recall_u = interseccion / len(relevants)
        recalls.append(recall_u)

    return sum(recalls) / len(recalls) if recalls else 0.0

recall_lightgcn = recall_at_k(top10_lightgcn, dic_items_relevantes, k=10)
print(f"Recall@10: {recall_lightgcn:.4f}")


Recall@10: 0.0625


Cálculo de HitRate@10:

In [ ]:
def hitrate_at_k(topk_dict, relevantes_dict, k=10):
    hitrates = []

    for user_id, relevants in relevantes_dict.items():

        if user_id not in topk_dict:
            continue

        topk_items = topk_dict[user_id][:k]

        interseccion = set(topk_items) & set(relevants)

        hitrate_u = 1 if len(interseccion) > 0 else 0
        
        hitrates.append(hitrate_u)

    return sum(hitrates) / len(hitrates) if hitrates else 0.0
    
hitrate_lightgcn = hitrate_at_k(top10_lightgcn, dic_items_relevantes, k=10)
print(f"HitRate@10: {hitrate_lightgcn:.4f}")


HitScore@10: 0.0645


Otra métrica que se utilizará es el F1 Score. Se calcula y guarda en una variable:

In [19]:
f1score_lightgcn = 2 * (precision_lightgcn * recall_lightgcn) / (precision_lightgcn + recall_lightgcn)
print(f"F1 Score@10: {f1score_lightgcn:.4f}")

F1 Score@10: 0.0117


Ahora, se calcula el MAP@K. Se comienzan por definir dos funciones base, una para calcular el Average Precision at K (AP@K) para un usuario y luego otra para calcular el MAP@K como el promedio del AP@K de todos los usuarios. Finalmente, se calcula el MAP@10:

In [20]:
def ap_at_k(items_relevantes, items_recomendados, k):
    if len(items_recomendados) > k:
        items_recomendados = items_recomendados[:k]
        
    score = 0
    num_hits = 0
    for i, p in enumerate(items_recomendados):
        if p in items_relevantes and p not in items_recomendados[:i]:
            num_hits += 1
            score += (num_hits / (i + 1))
    
    if not items_relevantes:
        return 0

    return score / min(len(items_relevantes), k)

def map_at_k(relevantes_dict, dict_recomendados, k):
    ap_scores = []
    for user_id, lista_recomendados in dict_recomendados.items():
        items_relevantes = relevantes_dict[user_id]
        ap = ap_at_k(items_relevantes, lista_recomendados, k)
        ap_scores.append(ap)
    
    return sum(ap_scores) / len(ap_scores)

map_lightgcn = map_at_k(dic_items_relevantes, top10_lightgcn, k=10)
print(f"MAP@10: {map_lightgcn:.4f}")

MAP@10: 0.0186


Ahora, se calcula la diversidad promedio de las recomendaciones (Diversity). Para esto se analizan los géneros de videojuegos recomendados y la métrica representa cuántos géneros distintos de videojuegos se recomiendan en promedio. Se comienza definiendo dos funciones para realizar los cálculos y luego se obtiene Diversity: 

In [21]:
#código basado en el elaborado por Nicolás Bueno y Felipe Fuentes en Tarea del curso.

#función para calcular la cantidad de géneros distintos de videojuegos en la lista de recomendación de un usuario
def diversity_user(items_recomendados, dict_item_genero):
    if not items_recomendados:
        return 0
    
    unique_generos = set()
    for item_id in items_recomendados:
        if item_id in dict_item_genero:
            #se recorre la lista de tags (géneros) asociados al videojuego
            for tag in dict_item_genero[item_id]:
                unique_generos.add(tag)

    return float(len(unique_generos))

#función para calcular la cantidad promedio de géneros distintos de videojuegos en todas las listas de recomendación generadas
def diversity(recomendaciones, dict_item_genero):
    total = 0
    cant_usuarios = 0
    for recs in recomendaciones.values():
        total += diversity_user(recs, dict_item_genero)
        cant_usuarios += 1
        
    return total / max(cant_usuarios, 1)

#los géneros de un videojuego se consideran como los "tags" asociados en el archivo de metadata (guardado en en el dataframe games_metadata)
dict_item_genero = dict(zip(games_metadata["app_id"].astype(int), games_metadata["tags"]))
diversity_lightgcn = diversity(top10_lightgcn, dict_item_genero)
print(f"Diversidad promedio: {diversity_lightgcn}")

Diversidad promedio: 25.230365435089844


En resumen, las métricas calculadas del modelo son:

In [ ]:
print("Resumen de las métricas:")
print()
print(f"Recall@10: {recall_lightgcn:.4f}")
print(f"Precision@10: {precision_lightgcn:.4f}")
print(f"F1 Score@10: {f1score_lightgcn:.4f}")
print(f"NDCG@10: {ndcg_lightgcn:.4f}")
print(f"HitRate@10: {hitrate_lightgcn:.4f}")
print(f"MAP@10: {map_lightgcn:.4f}")
print(f"Diversity: {diversity_lightgcn:.4f}")

Resumen de las métricas:

Recall@10: 0.0625
Precision@10: 0.0065
F1 Score@10: 0.0117
NDCG@10: 0.0279
HitScore@10: 0.0645
MAP@10: 0.0186
Diversity: 25.2304


## 5- Referencias

[1] Deng, K., He, X., Li, Y., Wang, M., Wang, X., & Zhang, Y. (2020). LightGCN: Simplifying and powering graph Convolution Network for recommendation. ArXiv. Obtenido de http://arxiv.org/abs/2002.02126

[2] Diapositivas de la clase "Clase de Evaluación: metricas de error y ranking", como apoyo para implementar cálculo de métricas. Enlace: https://github.com/PUC-RecSys-Class/RecSysPUC-2025-2/blob/master/clases/s3_c1-metricas_v3.pdf

[3] Documentación de RecBole 1.2.1: https://recbole.io/docs/

[4] Documentación modelo LightGCN de Recbole: https://recbole.io/docs/user_guide/model/general/lightgcn.html

[5] Repositorio Recbole en GitHub: https://github.com/RUCAIBox/RecBole

[6] Práctico de métricas del curso del semestre pasado, utilizado como base principalmente para programar las métrica de Diversity. Enlace: https://github.com/PUC-RecSys-Class/RecSysPUC-2025-2/blob/master/practicos/pr%C3%A1ctico_m%C3%A9tricas.ipynb

[7] Dataset principal: "Game Recommendations on Steam: A dataset of games, users and reviews for building recommendation systems". Anton Kozyriev, 2024. Kaggle. Link: https://www.kaggle.com/datasets/antonkozyriev/game-recommendations-on-steam?select=recommendations.csv

[8] Metadata adicional de videojuegos de Steam: "Steam Store Games (Clean dataset): Combined data of 27,000 games scraped from Steam and SteamSpy APIs". Nik Davis, 2019

[9] Uso de IA. Se consultó a ChatGPT sobre la librería RecBole, el uso de funciones y formatos de LightGCN, y errores. Link al chat: https://chatgpt.com/share/68ffb7df-f778-8008-b287-14268d81d203